# Deploying Flat-Bug

## Flat-bug setup

In [1]:
# Import the main model class
from flat_bug.predictor import Predictor

# Optionally import the set_log_level function to control the verbosity of flat-bug module
# from flat_bug import set_log_level
# set_log_level("DEBUG") # default is "INFO"

## Reproducible test setup
This part is only necessary if you don't have local test files (model weights/images) available. If you have them, you can skip this part.

In [2]:
import os
from urllib.request import urlretrieve

# We have sample weights and image available in a public repo
REMOTE_REPO = "https://anon.erda.au.dk/share_redirect/eU5VZ12Uzj/"

weights = "fb_YOLOv8M-seg.pt"
if not os.path.exists(weights):
    urlretrieve(REMOTE_REPO + "fb_YOLOv8M-seg.pt", weights)
image = "test_image.jpg"
if not os.path.exists(image):
    urlretrieve(REMOTE_REPO + "test_image.jpg", image)

## Usage

In [6]:
# Load the model
model = Predictor(weights, device="cuda:0", dtype="float16")

# Predict
prediction = model.pyramid_predictions(image)

for k, v in prediction.json_data.items():
    v = str(v)
    if len(v) > 100:
        v = v[:100] + "..."
    print(f"{k}: {v}")

YOLOv8m-seg summary (fused): 245 layers, 27,222,963 parameters, 0 gradients, 110.0 GFLOPs
boxes: [[2616, 1648, 2768, 1821], [915, 1589, 1105, 1802], [1394, 815, 1503, 961], [1511, 629, 1612, 767], ...
contours: [[[2707.33349609375, 2704.0, 2702.0, 2700.0, 2698.0, 2695.33349609375, 2683.33349609375, 2672.0, 267...
confs: [0.93505859375, 0.93359375, 0.9248046875, 0.92431640625, 0.91943359375, 0.91796875, 0.9140625, 0.913...
classes: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
scales: [1.0, 1.0, 0.892448874947216, 1.0, 0.892448874947216, 0.892448874947216, 1.0, 1.0, 1.0, 1.0, 1.0, 1....
image_path: test_image.jpg
image_width: 3840
image_height: 2160
mask_width: 3840
mask_height: 2160
identifier: None


## Plot results

In [4]:
from matplotlib import pyplot as plt

# Plot all the predictions
plt.figure(figsize=(20, 10))
plt.imshow(prediction.plot())
plt.gca().axis("off")
plt.title("Predictions")
plt.show()

In [5]:
# Plot the cropped images
n_display = min(64, len(prediction))
crops, masks, confidences = prediction.crops[:n_display], prediction.crop_masks[:n_display], prediction.confs[:n_display]

n_crops = len(crops)
ncol = 8
nrow = n_crops // ncol
if n_crops % ncol:
    nrow += 1

fig, axs = plt.subplots(nrow, ncol, figsize=(2.5 * ncol, 2.5 * nrow))
axs = axs.flatten() if n_crops > 1 else [axs]
for ax, crop, mask, conf in zip(axs, crops, masks, confidences):
    crop = crop.permute(1, 2, 0).cpu()
    mask = 1 - mask.squeeze(0).cpu().float()
    w, h = mask.shape
    if w < h:
        mask = mask.transpose(1, 0)
        crop = crop.transpose(1, 0)
    ax.imshow(crop)
    ax.imshow(mask, cmap="gray", vmin=0, vmax=1, alpha=mask * 0.5) 
    ax.set_title(f"Confidence: {conf * 100:.2f}%")
    ax.axis("off")
plt.suptitle("Cropped images")
plt.show()